# YOLOv8 Basketball Player Detection Training

This notebook trains a YOLOv8 model for detecting basketball players using bounding boxes derived from segmentation masks.

## 1. Setting Up Google Colab Environment

Connect to Google Colab, mount Google Drive if needed, and ensure GPU runtime is enabled for faster training.

In [ ]:
# Mount Google Drive (if your data is there)
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
import torch
print("GPU available:", torch.cuda.is_available())

# Change to your project directory (adjust path as needed)
import os
os.chdir('/content/drive/MyDrive/NBAYOLOTRACKER')  # Or wherever your folder is

## 2. Installing Required Libraries for YOLOv8

Install Ultralytics YOLOv8 library and other dependencies like OpenCV and PyTorch using pip.

In [ ]:
!pip install ultralytics opencv-python torch torchvision

## 3. Preparing the Dataset

Download or upload the dataset, organize it into train/val/test folders, and create a YAML configuration file for YOLOv8.

In [ ]:
# Run data conversion if not done
!python scripts/convert_masks_to_boxes.py

# Check the data
!ls data/yolo_data/images/train | head -5
!ls data/yolo_data/labels/train | head -5

## 4. Configuring YOLOv8 Model

Load a pre-trained YOLOv8 model (e.g., yolov8n.pt) and adjust hyperparameters like epochs, batch size, and image size.

In [ ]:
from ultralytics import YOLO

# Load pre-trained model
model = YOLO('yolov8n.pt')  # Nano model for speed

# Hyperparameters
epochs = 50
batch_size = 16
imgsz = 640

## 5. Training the Model

Run the training command using the model.train() method with the dataset and configuration.

In [ ]:
yaml_path = '/content/drive/MyDrive/NBAYOLOTRACKER/data/yolo_data/data.yaml'

yaml_content = """path: /content/drive/MyDrive/NBAYOLOTRACKER/data/yolo_data
train: images/train
val: images/val

nc: 1
names: ['player']
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(open(yaml_path).read())

# Train the model
results = model.train(
    data='/content/drive/MyDrive/NBAYOLOTRACKER/data/yolo_data/data.yaml',
    epochs=epochs,
    batch=batch_size,
    imgsz=imgsz,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

In [ ]:
from pathlib import Path

# Ensure the YOLO config is correct and paths are valid
root = Path.cwd()
yolo_dir = root / 'data' / 'yolo_data'
print('Current working dir:', root)
print('YOLO data dir:', yolo_dir)
print('Train images exist:', (yolo_dir / 'images' / 'train').exists())
print('Val images exist:', (yolo_dir / 'images' / 'val').exists())
print('YAML path:', yolo_dir / 'data.yaml')

yaml_content = '''
train: images/train
val: images/val

nc: 1
names: ['player']
'''
(yolo_dir / 'data.yaml').write_text(yaml_content)
print('Wrote correct data.yaml with relative paths')

## 6. Evaluating the Model

Validate the trained model on the validation set and compute metrics like mAP and precision.

In [ ]:
# Validate the model
metrics = model.val()
print("Validation metrics:", metrics)

## 7. Saving and Exporting the Trained Model

Save the best model weights and export them in formats like ONNX or TensorFlow for deployment.

In [ ]:
# Export the model
model.export(format='onnx')  # Or 'torchscript', 'tensorflow'

# The best weights are already saved in runs/detect/train/weights/best.pt